# Notebook #0:</br> Data Anonymisation
#### by Sebastian Einar Salas Røkholt

---

**Notebook Index**

- [**1 - Introduction and Notebook Setup**](#1---introduction-and-notebook-setup)
  - [*1.1 Setup*](#11-setup)
  - [*1.2 Load sensitive raw dataset*](#12-load-sensitive-raw-dataset)
- [**2 - Sensitive Feature Engineering**](#2---sensitive-feature-engineering)
  - [*2.1 Derive `minutes_elapsed` from `timestamp`*](#21-derive-minutes_elapsed-from-timestamp)
  - [*2.2 Attach ambient temperature from nearest weather station*](#22-attach-ambient-temperature-from-nearest-weather-station)
- [**3 - Build Public Dataset**](#3---build-public-dataset)

## 1 - Introduction and Notebook Setup

This notebook anonymises the sensitive charging-session extract into a public base dataset.
The workflow computes non-sensitive derived variables needed downstream, then removes
sensitive attributes before export.

### 1.1 Setup

In [1]:
import os
import requests
import re
import pickle as pkl
import pandas as pd
import numpy as np
from glob import glob
from io import StringIO
from datetime import datetime, timedelta
from haversine import haversine, Unit
from dotenv import load_dotenv
from typing import List, Tuple

load_dotenv(override=True)

# API constants
FROST_CLIENT_ID = os.getenv("FROST_API_CLIENT_ID")
DMI_MET_OBS_API_KEY = os.getenv("DMI_MET_OBS_API_KEY")
DMI_STATIONS_API = "https://dmigw.govcloud.dk/v2/metObs/collections/station/items"
SMHI_STATIONS_API = "https://opendata-download-metobs.smhi.se/api/version/latest/parameter/1/station.json"

# Paths
RAW_PARQUET = "../Data/etron55-charging-sessions-raw-nov-24.parquet"
OUT_PARQUET = "../Data/etron55-charging-sessions-public.parquet"

pd.options.mode.copy_on_write = True
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 60)

### 1.2 Load sensitive raw dataset

In [2]:
raw_columns = [
    "charging_id", "timestamp",
    "power", "soc", "energy",
    "charging_duration",
    "charger_category", "nominal_power",
    "lat", "lon",
]

df = pd.read_parquet(RAW_PARQUET, columns=raw_columns).copy()
print(f"Loaded raw rows: {len(df):,}")
print(f"Loaded raw sessions: {df['charging_id'].nunique():,}")

Loaded raw rows: 1,643,654
Loaded raw sessions: 62,422


## 2 - Sensitive Feature Engineering

### 2.1 Derive `minutes_elapsed` from `timestamp`

In [3]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["nominal_power"] = df["nominal_power"].astype(int)
df["charger_category"] = df["charger_category"].astype("category")
df = df.sort_values(["charging_id", "timestamp"]).reset_index(drop=True)

df["minutes_elapsed"] = (
    df.groupby("charging_id", observed=False)["timestamp"]
    .transform(lambda s: (s - s.min()).dt.total_seconds() // 60)
    .astype(int)
)

print("Derived minutes_elapsed from timestamp.")

Derived minutes_elapsed from timestamp.


### 2.2 Attach ambient temperature from nearest weather station

In [4]:
df["lat"] = df["lat"].round(2)
df["lon"] = df["lon"].round(2)

# Converts the timestamp to the Frost-specified format, with granularity of 1 hour
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['timestamp_H'] = df.groupby('charging_id')['timestamp'].transform('first')
df['timestamp_H'] = df['timestamp_H'].dt.strftime('%Y-%m-%dT%H')
df['timestamp_d'] = df.groupby('charging_id')['timestamp'].transform('first')
df['timestamp_d'] = df['timestamp_d'].dt.strftime('%Y-%m-%d')
df.head()

print("Prepared geospatial and hourly helper columns for temperature retrieval.")

Prepared geospatial and hourly helper columns for temperature retrieval.


In [5]:
def calc_coll_period_per_location(df: pd.DataFrame) -> pd.DataFrame:
    # Get unique location-timestamp combinations, with time granularity per hour
    locations_timestamps_per_hour_df = df[["lat", "lon", "timestamp_d"]].drop_duplicates().sort_values(by=['lat', 'lon']).reset_index(drop=True)
    locations_grouped = locations_timestamps_per_hour_df.groupby(["lat", "lon"])
    # Calculate the data collection period per location
    locations_w_coll_periods = locations_grouped.agg(
        period_start=('timestamp_d', 'min'),
        period_end=('timestamp_d', 'max')
    ).reset_index()
    return locations_w_coll_periods

def NO_station_has_hourly_data(station_id: str, collection_period: str) -> bool:
    response = requests.get('https://frost.met.no/observations/availableTimeSeries/v0.jsonld',
        params={
            'sources': {station_id},
            'referencetime': collection_period, 
            'elements': 'air_temperature', 
            'timeresolutions': 'PT1H',
        },
        auth=(FROST_CLIENT_ID, '')
    )
    if response.status_code == 200:
        return True
    else:
        return False
    

def get_nearest_station(lat: float, lon: float, period_start: str, period_end: str, k=30) -> str:
    """Retrieves the station IDs of the k nearest weather stations 
    from a fixed point via MET Norway's Frost API. We retrieve the ids for multiple stations in case
    the <k nearest are new and we get an error when fetching the historical temperature from the Frost API later. 

    Args:
        latitude (float): The latitude
        longitude (float): The longitude
        api_client_id (str): The client ID for the Frost API. 
                             An ID can be created for free here: https://frost.met.no/auth/requestCredentials.html
        k (int, optional): The number of station IDs to check. Defaults to 30.

    Returns:
        List[str]: A list of IDs of the nearest weather stations, sorted (increasingly) by distance.
    """
    collection_period = f"{period_start}/{period_end}"
    response = requests.get('https://frost.met.no/sources/v0.jsonld',
        params={
            'geometry': f'nearest(POINT({lon} {lat}))',  # FROST API uses WKT specification, which requires lon, lat ordering
            'nearestmaxcount': k,
            'validtime': collection_period, 
            'elements': 'air_temperature'  # Filter by stations that collected temperature data
        },
        auth=(FROST_CLIENT_ID, '')
    )
    if response.status_code in [404, 412]:
        print(f"Could not find any weather stations for location with lat: {lat}, lon: {lon} \
            in time period {collection_period}")
        return np.nan, np.nan, np.nan
    else: 
        for station_metadata in response.json()["data"]:
            try:
                country = station_metadata["country"].lower()
                if country == "norge":
                    station_id, distance = station_metadata["id"], round(station_metadata["distance"], 1)
                    if NO_station_has_hourly_data(station_id, collection_period):
                        return country, station_id, distance
                    else:
                        continue

                elif country == "sverige":
                    station_id, distance = get_nearest_SE_station(lat, lon, period_start, period_end)
                    return country, station_id, distance
                elif country == "danmark":
                    station_id, distance = get_nearest_DK_station(lat, lon, period_start, period_end)
                    return country, station_id, distance
                else: 
                    print(f"Nearest weather station is in {country}, returning np.nan")
                    station_id, distance = np.nan, np.nan
                    return country, station_id, distance
        
            except (KeyError, IndexError, ValueError) as err:
                print(f"Encountered an error when attempting to parse result for lat: {lat}, lon: {lon} \
                    and collection period {collection_period}. Error message:\n {err}")
                print(f"Response from FROST API: {response.json()['data']}")
                print(f"\nWas attempting to parse this element when the error occurred: ")
                print(station_metadata)
                return np.nan, np.nan, np.nan
        print(f"Could not find a weather station with temperature recordings per hour among the closest {k}"\
              f"stations for lat: {lat}, lon: {lon} and collection period {collection_period}."\
              f"The nearest weather station (without hourly temperature data) was {response.json()["data"][0]["id"]}")
        return np.nan, np.nan, np.nan


# Convert date string to epoch
def date_to_epoch(date_str):
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    return int(dt.timestamp())


# Fetch all Swedish weather stations that have collected temperature data
response = requests.get(SMHI_STATIONS_API)
response.raise_for_status()
SE_stations = response.json().get('station', [])
# Fetch all Danish weather stations 
min_datetime = df["timestamp"].min().strftime("%Y-%m-%dT%H:%M:%SZ")
max_datetime = df["timestamp"].max().strftime("%Y-%m-%dT%H:%M:%SZ")
dk_station_api_params = {'api-key': DMI_MET_OBS_API_KEY, "type": "Synop", 
                         "datetime": f"{min_datetime}/{max_datetime}", "status": "Active"}
response = requests.get(DMI_STATIONS_API, params=dk_station_api_params)
response.raise_for_status()
DK_stations = response.json().get('features', [])


def get_SE_download_urls(station_id: str|int, return_metadata: bool=False) -> List[Tuple[str, List[str]]]:
    download_urls = []
    metadatas = []
    for period in ["corrected-archive", "latest-months"]:
        station_metadata_baseurl = "https://opendata-download-metobs.smhi.se/api/version/1.0/parameter/1/station/{station_id}/period/{period}.json"
        response = requests.get(station_metadata_baseurl.format(station_id=station_id, period=period))
        if response.status_code == 200: 
            station_metadata = response.json()
            metadatas.append(station_metadata)
            # print(f"Station metadata: {station_metadata}")
            data_download_info = station_metadata["data"]
            for link_info in data_download_info:
                download_urls.append((period, [l["href"] for l in link_info["link"] if l["href"].endswith((".csv", ".json"))]))
    if return_metadata:
        return download_urls, metadatas
    else: 
        return download_urls


def get_nearest_SE_station(lat: float, lon: float, period_start: str, period_end: str, all_SE_weather_stations: List[dict]=SE_stations):
    period_start_epoch = date_to_epoch(period_start)
    period_end_epoch = date_to_epoch(period_end)
    nearest_station = None
    min_distance = float('inf')
    for station in all_SE_weather_stations:
        station_id = station["id"]
        station_lat = station['latitude']
        station_lon = station['longitude']
        distance = haversine((lat, lon), (station_lat, station_lon), unit=Unit.KILOMETERS)
        if distance < min_distance:
            # Get more station metadata for info about data availability
            _, station_metadata = get_SE_download_urls(station_id, return_metadata=True)
            active_from = float("inf")
            active_to = -float("inf")
            for metadata in station_metadata:
                data_from = metadata["from"] // 1000
                data_to = metadata["to"] // 1000
                active_from = min(active_from, data_from)
                active_to = max(active_to, data_to)
            # Check that the station collected data in the specified period
            if period_start_epoch >= active_from and period_end_epoch <= active_to:
                    min_distance = distance  # all checks passed, update nearest station
                    nearest_station = {
                            'id': station_id,
                            'name': station["name"],
                            'lat': station_lat,
                            'lon': station_lon,
                            'distance': distance
                        }
    if nearest_station:
        # print(f"The nearest Swedish station from lat: {lat} lon: {lon} is " \
        #       f"{nearest_station['id']}: {nearest_station['name']} at {nearest_station['lat']}, "\
        #       f"{nearest_station['lon']}, which is {round(nearest_station['distance'], 1)}km away")
        return nearest_station['id'], round(nearest_station["distance"], 1)
    else:
        print(f"No Swedish weather station found for lat: {lat}, lon: {lon} in the period {period_start} to {period_end}")
        return np.nan, np.nan

def get_nearest_DK_station(lat: float, lon: float, period_start: str, period_end: str, 
                           all_DK_weather_stations: List[dict]=DK_stations):
    # Convert period to epoch
    period_start_epoch = date_to_epoch(period_start)
    period_end_epoch = date_to_epoch(period_end)
    nearest_station = None
    min_distance = float('inf')

    for station in all_DK_weather_stations:
        station_properties = station.get('properties', {})
        station_geometry = station.get('geometry', {}).get('coordinates', [])

        if not station_geometry:
            continue

        station_lon, station_lat = station_geometry

        # Extract active periods
        active_from = station_properties.get('operationFrom', float('inf'))
        active_to = station_properties.get('operationTo', None)
        if active_to in [None, "None"]:
            active_to = float('inf')
        active_from_epoch = date_to_epoch(active_from[:10]) if isinstance(active_from, str) else active_from
        active_to_epoch = date_to_epoch(active_to[:10]) if isinstance(active_to, str) else active_to

        # Check if the station was active during the requested period
        if period_start_epoch >= active_from_epoch and period_end_epoch <= active_to_epoch:
            if "temp_dry" in station_properties.get("parameterId"):
                distance = haversine((lat, lon), (station_lat, station_lon), unit=Unit.KILOMETERS)
                if distance < min_distance:
                    min_distance = distance
                    nearest_station = {
                        'id': station_properties.get('stationId'),
                        'name': station_properties.get('name'),
                        'lat': station_lat,
                        'lon': station_lon,
                        'distance': distance
                    }

    if nearest_station:
        # print(f"The nearest Danish station from lat: {lat}, lon: {lon} is "
        #       f"{nearest_station['id']}: {nearest_station['name']} at {nearest_station['lat']}, {nearest_station['lon']}, "
        #       f"which is {round(nearest_station['distance'], 1)}km away")
        return nearest_station['id'], round(nearest_station["distance"], 1)
    else:
        print(f"No Danish station found for lat: {lat}, lon: {lon} in the period {period_start} to {period_end}")
        return np.nan, np.nan

In [6]:
# Run calculations or retrieve pre-computed values if available
locations_df_pkl_path = "../Data/charging_locations_df.pkl"
if os.path.exists(locations_df_pkl_path):
        with open(locations_df_pkl_path, "rb") as locations_df_obj:
            charging_locations_df = pkl.load(locations_df_obj)
else: 
    charging_locations_df = calc_coll_period_per_location(df)
    charging_locations_df[["country", "nearest_weather_station", "distance_to_weather_station_in_km"]] = charging_locations_df.apply(
        lambda row: pd.Series(get_nearest_station(row["lat"], row["lon"], row["period_start"], row["period_end"])), 
        axis=1)
    charging_locations_df["nearest_weather_station"] = charging_locations_df["nearest_weather_station"].astype(str)
    with open(locations_df_pkl_path, "wb") as locations_df_obj:
        pkl.dump(charging_locations_df, locations_df_obj)


print(f"Location cache rows: {len(charging_locations_df):,}")

Location cache rows: 286


In [7]:
stations_per_period = charging_locations_df.copy()
stations_per_period['period_start'] = pd.to_datetime(stations_per_period['period_start'], format="%Y-%m-%d")
stations_per_period['period_end'] = pd.to_datetime(stations_per_period['period_end'], format="%Y-%m-%d")
stations_per_period = stations_per_period.sort_values(['nearest_weather_station', 'period_start'])

def merge_periods(group):
    group = group.sort_values('period_start')
    merged = []
    for _, row in group.iterrows():
        if not merged or merged[-1]['period_end'] < row['period_start']:
            merged.append(row.to_dict())
        else:
            merged[-1]['period_end'] = max(merged[-1]['period_end'], row['period_end'])
    return pd.DataFrame(merged)

stations_with_merged_coll_periods = stations_per_period.groupby('nearest_weather_station')[["nearest_weather_station", "country", "period_start", "period_end"]].apply(merge_periods).reset_index(drop=True)

print("Total number of weather stations we need to pull temperature data from:")
display(stations_with_merged_coll_periods["country"].value_counts())
print(f"The max distance to the nearest weather station is {charging_locations_df["distance_to_weather_station_in_km"].max()}km")

def download_NO_temperatures_in_period(station_id: str, period_start: pd.Timestamp, 
                          period_end: pd.Timestamp, download_file_path: str, 
                          api_client_id: str) -> None:
    # Add time to dates to capture the entirety of the last day in period
    collection_period = f"{period_start.strftime("%Y-%m-%dT%H:%M:%S")}/{period_end.strftime("%Y-%m-%dT%H:%M:%S")}"
    # print(f"Fetching temperatures for {station_id} in collection interval"\
    #       f" {collection_period} from MET Norway's FROST API...")
    response = requests.get(
        'https://frost.met.no/observations/v0.jsonld',
        params={
            'sources': station_id,
            'elements': 'air_temperature',
            'referencetime': collection_period,
            'timeresolutions': 'PT1H',  # Fetch 1 value per hour
        },
        auth=(api_client_id, '')
    )
    
    if response.status_code == 200:
        timestamps = []
        temperatures = []
        try:
            data = response.json()["data"]
            for obs in data:
                timestamps.append(obs["referenceTime"])
                temperatures.append(obs["observations"][0]["value"])
            # print(f"Successfully retrieved {len(temperatures)} temperature readings.")
            # save results
            temps_df = pd.DataFrame({"timestamp": timestamps, "temperature": temperatures})
            with open(download_file_path, "wb") as f:
                pkl.dump(temps_df, f)
                # print(f"Successfully downloaded temperatures for station ID {station_id}\
                #       \nin collection period {period_start.day} --> {period_end.day} to file {download_file_path}")
            
        except (TypeError, KeyError) as err:
            print(f"Error in download_NO_temperatures_in_period: Could not parse response from Frost API. {err}")
            print(f"Response: {response.json()}")
            raise err
    elif response.status_code == 412 or response.status_code == 404:
        print(f"No (or insufficient number of) temperature values found for station {station_id}\n\
               in collection period {period_start.day} --> {period_end.day}")
    else:
        print(f"Error in download_temperatures: Encountered a {response.status_code} error from the Frost API:\n\
               {response.json()["error"]}")


def download_SE_all_temperatures(station_id: str, download_path: str) -> None:
    all_dfs = []
    download_urls = get_SE_download_urls(station_id)
    if download_urls:
        # print(download_urls)
        for period, urls in download_urls:
            # print(url)
            for url in urls:
                file_ext = os.path.splitext(url)[1]
                if file_ext == ".json":
                    response = requests.get(url)
                    response.raise_for_status()
                    data = response.json()
                    observations = data.get('value', [])
                    if observations:
                        obs_df = pd.DataFrame(observations)
                        # Convert 'date' from milliseconds to datetime
                        obs_df['timestamp'] = pd.to_datetime(obs_df['date'], unit='ms')
                        obs_df.rename(columns={'value': 'temperature'}, inplace=True)
                        obs_df = obs_df[['timestamp', 'temperature']]
                        obs_df['temperature'] = obs_df['temperature'].str.replace(',', '.').astype(float)
                        # Performs linear interpolation with time indexing
                        obs_df.set_index('timestamp', inplace=True)
                        obs_df = obs_df.loc[~obs_df.index.duplicated(), :]  # drop duplicate indeces, if any
                        obs_df = obs_df.resample('h').interpolate(method='time')
                        obs_df.reset_index(inplace=True)
                        all_dfs.append(obs_df)
                        # print(f"DataFrame result for {file_ext} from period {period}:")
                        # display(obs_df.tail(10))
                        break
                    else:
                        print(f"No JSON observations found for period {period}")
                    
                elif file_ext == ".csv":
                    response = requests.get(url)
                    response.raise_for_status()
                    # Skips non-data info at the beginning of the .csv
                    csv_text = response.text
                    data_headers = "Datum;Tid (UTC);Lufttemperatur"
                    csv_text_data = csv_text[csv_text.find(data_headers):]
                    csv_data = StringIO(csv_text_data)
                    obs_df = pd.read_csv(csv_data, sep=';', low_memory=False)
                    obs_df.drop(["Kvalitet", "Unnamed: 4", "Tidsutsnitt:"], axis=1, inplace=True, errors="ignore")
                    obs_df['timestamp'] = pd.to_datetime(obs_df['Datum'] + ' ' + obs_df['Tid (UTC)'], format='%Y-%m-%d %H:%M:%S')
                    obs_df.rename(columns={"Lufttemperatur": 'temperature'}, inplace=True)
                    obs_df = obs_df[['timestamp', 'temperature']]
                    # Performs linear interpolation with time indexing
                    obs_df.set_index('timestamp', inplace=True)
                    obs_df = obs_df.loc[~obs_df.index.duplicated(), :]  # drop duplicate indeces, if any
                    obs_df = obs_df.resample('h').interpolate(method='time')
                    obs_df.reset_index(inplace=True)
                    all_dfs.append(obs_df)


    # Concatenate the DataFrames with data from the corrected archive and from latest months
    temps_df = pd.concat(all_dfs, ignore_index=True)
    # If duplicates, keep values from corrected archive
    temps_df = temps_df.drop_duplicates(subset='timestamp', keep='first')  

    # Save the DataFrame with all data for the station
    with open(download_path, "wb") as f:
        pkl.dump(temps_df, f)
        # print(f"Successfully downloaded all temperatures for station ID {station_id} to file {download_path}")
    
def download_DK_temperatures_in_period(station_id: str, period_start: pd.Timestamp, 
                          period_end: pd.Timestamp, download_file_path: str, api_key: str) -> None:
    collection_period = f"{period_start.strftime("%Y-%m-%dT%H:%M:%SZ")}/{period_end.strftime("%Y-%m-%dT%H:%M:%SZ")}"
    print(f"Fetching temperatures for Danish weather station {station_id} in collection interval"\
          f" {collection_period} from the DMI API...")
    response = requests.get(
        'https://dmigw.govcloud.dk/v2/metObs/collections/observation/items',
        params={
            'datetime': collection_period,
            'limit': '300000', # Set to max nr of measurements allowed
            'stationId': station_id,
            'parameterId': 'temp_dry',  # Dry temperature
            'api-key': api_key,
        },
    )
    
    if response.status_code == 200:
        timestamps = []
        temperatures = []
        try:
            data = response.json().get("features", [])
            for obs in data:
                obs_props = obs["properties"]
                timestamps.append(obs_props["observed"])
                temperatures.append(obs_props["value"])
            print(f"Successfully retrieved {len(temperatures)} temperature readings.")
            temps_df = pd.DataFrame({"timestamp": timestamps, "temperature": temperatures})
            # DMI collects temp data per 10 min. Resample to per hour (mean)
            temps_df["timestamp"] = pd.to_datetime(temps_df["timestamp"])
            temps_df = temps_df.set_index("timestamp")
            temps_df = temps_df.resample("h").mean().round()  # Round temperature to nearest int
            with open(download_file_path, "wb") as f:
                pkl.dump(temps_df, f)
                print(f"Successfully downloaded temperatures for station ID {station_id}\
                      \nin collection period {period_start} --> {period_end} to file {download_file_path}")
            
        except (TypeError, KeyError) as err:
            print(f"Error in download_DK_temperatures_in_period: Could not parse response from DMI API. {err}")
            print(f"Response: {response.json()}")
            raise err
    elif response.status_code == 412 or response.status_code == 404:
        print(f"No (or insufficient number of) temperature values found for station {station_id}\n\
               in collection period {period_start} --> {period_end}")
    else:
        print(f"Error in download_temperatures: Encountered a {response.status_code} error from the DMI API:\n\
               {response.json()}")

Total number of weather stations we need to pull temperature data from:


country
norge      151
sverige     46
danmark      5
Name: count, dtype: int64

The max distance to the nearest weather station is 44.3km


In [8]:
for _, row in stations_with_merged_coll_periods.iterrows():
    station_id = row["nearest_weather_station"]
    period_start = row["period_start"]
    # The timestamps' time is set to 00:00:00 by default, so we need to extend them 
    # to capture the entirety of the last day in the collection period
    period_end = row["period_end"] + timedelta(hours=23)
    country = row["country"]
    download_file_path = f"../Data/temperature_data/{station_id}_temp_from_{period_start.date()}_to_{period_end.date()}_df.pkl"
    
    if os.path.exists(download_file_path):
        continue
        # print(f"Already downloaded temperatures for station ID {station_id} from {period_start.date()} to {period_end.date()} to file {download_file_path}")

    else:
        if country == "norge":
            download_NO_temperatures_in_period(station_id, period_start, period_end, download_file_path, api_client_id=FROST_CLIENT_ID)
        elif country == "sverige":
            # Path to save all data for the station
            all_station_temp_data_path = f"../Data/temperature_data/SE_stations/{station_id}_all_temp_data.pkl"
            if not os.path.exists(all_station_temp_data_path):
                # Download all data for the station if not already done
                download_SE_all_temperatures(station_id, all_station_temp_data_path)
            # Load the station's data
            with open(all_station_temp_data_path, "rb") as f:
                temps_df = pkl.load(f)
                # Filter data for the desired period
                period_start_dt = pd.to_datetime(period_start)
                period_end_dt = pd.to_datetime(period_end)
                indeces_in_period = (temps_df["timestamp"] >= period_start_dt) & (temps_df["timestamp"] <= period_end_dt)
                filtered_df = temps_df.loc[indeces_in_period]
                if not filtered_df.empty:
                    # Save the filtered data
                    with open(download_file_path, "wb") as f:
                        pkl.dump(filtered_df, f)
                    # print(f"Successfully saved temperatures for station ID {station_id} from {period_start.date()} to {period_end.date()} to file {download_file_path}")
                else:
                    print(f"No temperature data available for station {station_id} in the specified collection period {period_start.date()} to {period_end.date()}.")
        elif country == "danmark":
            download_DK_temperatures_in_period(station_id, period_start, period_end, download_file_path, api_key=DMI_MET_OBS_API_KEY)

print("Temperature retrieval loop finished.")

Temperature retrieval loop finished.


In [9]:
df_combined = pd.merge(df, charging_locations_df[['lat', 'lon', 'nearest_weather_station']], on=['lat', 'lon'], how='left')

# Retrieves the start time (hour) per charging session
session_start = df_combined.groupby("charging_id", as_index=False, observed=False).first()[["charging_id", "timestamp_H", "nearest_weather_station"]]
session_start["timestamp_H"] = pd.to_datetime(session_start["timestamp_H"])
session_start.rename(columns={"timestamp_H": "timestamp"}, inplace=True)

temp_files = glob("../Data/temperature_data/*_temp_from_*_to_*_df.pkl")
# Create a dictionary with DataFrame objects, i.e. {station_id: temp_df}
station_dfs = {}
for fname in temp_files:
    # Assuming file name like 'SN76914_temp_from_2020-03-01_to_2024-10-30_df.pkl'
    match = re.search(pattern=r".*temperature_data/(.*?)_temp_from_.*", string=fname)
    station_id = match.group(1)  
    temp_df = pd.read_pickle(fname)
    try:
        temp_df['timestamp'] = pd.to_datetime(temp_df['timestamp'])
    except KeyError as err: 
        temp_df.reset_index(inplace=True)
        temp_df['timestamp'] = pd.to_datetime(temp_df['timestamp'])
    temp_df["timestamp"] = temp_df["timestamp"].dt.tz_localize(None)
    temp_df.sort_values('timestamp', inplace=True)
    station_dfs[station_id] = temp_df

# For each station, we find the temperature at the session's start time.
results = []
for station_id, group in session_start.groupby('nearest_weather_station'):
    if station_id in station_dfs:
        temp_df = station_dfs[station_id]
        # Performs an as-of merge to find the closest temperature reading before/at session start
        merged = pd.merge_asof(
            group.sort_values('timestamp'),
            temp_df, 
            on='timestamp',
            direction='nearest'
        )
        results.append(merged)
    else:
        print(f"No temperature file found for station with ID {station_id}")
        group['temperature'] = None
        results.append(group)

# contains a 'temperature' column aligned with each charging_id
session_start_temps = pd.concat(results, ignore_index=True)

# Merges session_start_temps back to the main timeseries dataset
df = pd.merge(
    df_combined, 
    session_start_temps[['charging_id', 'temperature']], 
    on='charging_id', 
    how='left'
)

# Rounds and renames the temperature column
df["temperature"] = df["temperature"].apply(lambda x: int(round(x, 0)))
df = df.rename(columns={"temperature": "temp"})

print("Merged session-start temperatures into the anonymisation dataframe.")

Merged session-start temperatures into the anonymisation dataframe.


## 3 - Build Public Dataset

In [10]:
public_columns = [
    "charging_id", "minutes_elapsed", "power", "soc", "energy",
    "charging_duration", "charger_category", "nominal_power", "temp",
]

df_public = df[public_columns].copy()
df_public = df_public.sort_values(["charging_id", "minutes_elapsed"]).reset_index(drop=True)
df_public.to_parquet(OUT_PARQUET, index=False)

print(f"Saved anonymised public dataset to {OUT_PARQUET}")
print(f"Rows: {len(df_public):,}")
print(f"Sessions: {df_public['charging_id'].nunique():,}")
print(f"Columns: {list(df_public.columns)}")

Saved anonymised public dataset to ../Data/etron55-charging-sessions-public.parquet
Rows: 1,643,654
Sessions: 62,422
Columns: ['charging_id', 'minutes_elapsed', 'power', 'soc', 'energy', 'charging_duration', 'charger_category', 'nominal_power', 'temp']
